# Exploratory Data Analysis
**UCI Default of Credit Card Clients**

Goals:
- Understand the data shape, types, and quality
- Examine class balance
- Explore feature distributions and correlations
- Identify preprocessing needs
- Surface candidate predictive features

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
pd.set_option('display.max_columns', 30)

In [ ]:
from src.ingestion.load_data import load_raw
df = load_raw()
print(f'Shape: {df.shape}')
df.head()

## 1. Basic info

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

## 2. Class balance

In [ ]:
target = 'default_payment_next_month'
counts = df[target].value_counts()
print(counts)
print(f'\nDefault rate: {df[target].mean():.2%}')

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot(kind='bar', ax=ax, color=['steelblue', 'salmon'])
ax.set_xticklabels(['No Default', 'Default'], rotation=0)
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/figures/class_balance.png', dpi=150)
plt.show()

## 3. Feature distributions

In [ ]:
numeric_cols = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, numeric_cols):
    df[col].hist(bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=150)
plt.show()

## 4. Categorical features

In [ ]:
cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, cat_cols):
    default_rate = df.groupby(col)[target].mean()
    default_rate.plot(kind='bar', ax=ax, color='salmon')
    ax.set_title(f'Default rate by {col}')
    ax.set_ylabel('Default rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('../reports/figures/default_by_category.png', dpi=150)
plt.show()

## 5. Payment status vs default rate

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
dr = df.groupby('PAY_0')[target].mean()
dr.plot(kind='bar', ax=ax, color='salmon')
ax.set_title('Default rate by PAY_0 (most recent payment status)')
ax.set_xlabel('PAY_0 status')
ax.set_ylabel('Default rate')
plt.tight_layout()
plt.show()

## 6. Correlation heatmap

In [ ]:
corr_cols = ['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2', 'BILL_AMT1', 'PAY_AMT1', target]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', ax=ax, cmap='coolwarm', center=0)
ax.set_title('Feature Correlations')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150)
plt.show()

## 7. Key observations

- **~22% default rate** → significant class imbalance requiring `class_weight` or `scale_pos_weight`
- **PAY_0** has a strong monotonic relationship with default rate — most important feature
- **LIMIT_BAL** is negatively correlated with default (higher limit → lower default)
- **EDUCATION** and **MARRIAGE** show some variation but modest effect sizes
- BILL_AMT columns are highly correlated with each other — derived trend features will be more informative
- No missing values in this dataset